# 01 - Data Audit and Business Understanding

**Goal:** inspect each source independently before creating any relationship between systems.

This notebook verifies file availability, columns, data types, missing values, duplicates, and candidate keys. It also records the integration rules used throughout the project.

| Source | Business purpose | Integration rule |
|---|---|---|
| Olist | Customer and transaction source of truth | Use `customer_unique_id` as the canonical customer key |
| Cosmetics clickstream | Behavioral signals | Use an explicitly simulated identity-resolution mapping |
| Datafiniti/Amazon | External product sentiment | Aggregate at product/category level; never join to Olist customers |
| Synthetic campaigns | Marketing demonstration | Mark every generated event as synthetic |

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", palette="Set2")


def find_project_root() -> Path:
    """Find the project root whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter from the project root or notebooks directory.")


ROOT = find_project_root()
DATASETS = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print("Project root located.")

Project root located.


## 1. File inventory

Large files are sampled for the quick audit so the notebook remains responsive. The ETL notebook reads only the columns it needs.

In [2]:
expected_files = {
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "product_category_name_translation.csv",
    "ecommerce_events_2019_dec.csv",
    "amazon_consumer_reviews.csv",
}

available_files = {path.name for path in DATASETS.glob("*.csv")}
missing_files = sorted(expected_files - available_files)
assert not missing_files, f"Missing required datasets: {missing_files}"

inventory = pd.DataFrame([
    {
        "file": path.name,
        "size_mb": round(path.stat().st_size / 1024**2, 2),
    }
    for path in sorted(DATASETS.glob("*.csv"))
])
display(inventory)

,file,size_mb
0,amazon_consumer_reviews.csv,46.72
1,ecommerce_events_2019_dec.csv,399.43
2,olist_customers_dataset.csv,8.71
3,olist_order_items_dataset.csv,14.83
4,olist_order_payments_dataset.csv,5.61
5,olist_order_reviews_dataset.csv,13.78
6,olist_orders_dataset.csv,16.93
7,olist_products_dataset.csv,2.30
8,product_category_name_translation.csv,0.00


## 2. Sample-level quality profile

The sample profile identifies obvious schema and quality problems. Exact row-level validation is performed for the warehouse tables during ETL.

In [3]:
audit_rows = []
for path in sorted(DATASETS.glob("*.csv")):
    sample = pd.read_csv(path, nrows=10_000, low_memory=False)
    audit_rows.append({
        "file": path.name,
        "sample_rows": len(sample),
        "columns": sample.shape[1],
        "duplicate_sample_rows": int(sample.duplicated().sum()),
        "missing_cells": int(sample.isna().sum().sum()),
        "missing_pct": round(100 * sample.isna().mean().mean(), 2),
    })

audit_summary = pd.DataFrame(audit_rows).sort_values("file")
display(audit_summary)

,file,sample_rows,columns,duplicate_sample_rows,missing_cells,missing_pct
0,amazon_consumer_reviews.csv,10000,21,0,41531,19.78
1,ecommerce_events_2019_dec.csv,10000,9,557,14279,15.87
2,olist_customers_dataset.csv,10000,5,0,0,0.00
3,olist_order_items_dataset.csv,10000,7,0,0,0.00
4,olist_order_payments_dataset.csv,10000,5,0,0,0.00
5,olist_order_reviews_dataset.csv,10000,7,0,14630,20.90
6,olist_orders_dataset.csv,10000,8,0,467,0.58
7,olist_products_dataset.csv,10000,9,0,756,0.84
8,product_category_name_translation.csv,71,2,0,0,0.00


## 3. Data dictionaries

A dictionary is exported for every source. These files are useful evidence of data understanding in a portfolio review.

In [4]:
for path in sorted(DATASETS.glob("*.csv")):
    sample = pd.read_csv(path, nrows=10_000, low_memory=False)
    dictionary = pd.DataFrame({
        "column": sample.columns,
        "inferred_dtype": [str(sample[column].dtype) for column in sample.columns],
        "missing_pct_sample": [round(100 * sample[column].isna().mean(), 2) for column in sample.columns],
        "unique_values_sample": [sample[column].nunique(dropna=True) for column in sample.columns],
        "example_value": [sample[column].dropna().iloc[0] if sample[column].notna().any() else None for column in sample.columns],
    })
    dictionary.to_csv(PROCESSED / f"dictionary_{path.stem}.csv", index=False)
    print(f"\n{path.name}")
    display(dictionary)


amazon_consumer_reviews.csv


,column,inferred_dtype,missing_pct_sample,unique_values_sample,example_value
0,id,str,0.00,10,AVqkIhwDv8e3D1O-lebb
1,name,str,0.00,11,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
2,asins,str,0.00,10,B01AHB9CN2
3,brand,str,0.00,1,Amazon
4,categories,str,0.00,10,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."
5,keys,str,0.00,10,"841667104676,amazon/53004484,amazon/b01ahb9cn2..."
6,manufacturer,str,0.00,1,Amazon
7,reviews.date,str,0.09,660,2017-01-13T00:00:00.000Z
8,reviews.dateAdded,str,6.42,1173,2017-07-03T23:33:15Z
9,reviews.dateSeen,str,0.00,279,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z"



ecommerce_events_2019_dec.csv


,column,inferred_dtype,missing_pct_sample,unique_values_sample,example_value
0,event_time,str,0.00,7009,2019-12-01 00:00:00 UTC
1,event_type,str,0.00,4,remove_from_cart
2,product_id,int64,0.00,4441,5712790
3,category_id,int64,0.00,316,1487580005268456287
4,category_code,str,98.57,7,furniture.bathroom.bath
5,brand,str,44.13,150,f.o.x
6,price,float64,0.00,724,6.27
7,user_id,int64,0.00,1450,576802932
8,user_session,str,0.09,2410,51d85cb0-897f-48d2-918b-ad63965c12dc



olist_customers_dataset.csv


...

,column,inferred_dtype,missing_pct_sample,unique_values_sample,example_value
0,order_id,str,0.00,10000,e481f51cbdc54678b7cc49136f2d6af7
1,customer_id,str,0.00,10000,9ef432eb6251297304e76186b10a928d
2,order_status,str,0.00,7,delivered
3,order_purchase_timestamp,str,0.00,9995,2017-10-02 10:56:33
4,order_approved_at,str,0.21,9868,2017-10-02 11:07:15
5,order_delivered_carrier_date,str,1.66,9511,2017-10-04 19:55:00
6,order_delivered_customer_date,str,2.80,9710,2017-10-10 21:25:13
7,order_estimated_delivery_date,str,0.00,421,2017-10-18 00:00:00



olist_products_dataset.csv


,column,inferred_dtype,missing_pct_sample,unique_values_sample,example_value
0,product_id,str,0.00,10000,1e9e8ef04dbcff4541ed26657ea517e5
1,product_category_name,str,1.88,72,perfumaria
2,product_name_lenght,float64,1.88,59,40.0
3,product_description_lenght,float64,1.88,2140,287.0
4,product_photos_qty,float64,1.88,17,1.0
5,product_weight_g,float64,0.01,1203,225.0
6,product_length_cm,float64,0.01,96,16.0
7,product_height_cm,float64,0.01,95,10.0
8,product_width_cm,float64,0.01,83,14.0



product_category_name_translation.csv


,column,inferred_dtype,missing_pct_sample,unique_values_sample,example_value
0,product_category_name,str,0.0,71,beleza_saude
1,product_category_name_english,str,0.0,71,health_beauty


## 4. Key checks and integration decisions

Olist's `customer_id` identifies the customer record attached to one order. Repeat-customer analytics must use `customer_unique_id`. This distinction is essential for valid frequency, repeat rate, churn, and CLV features.

In [4]:
customers = pd.read_csv(DATASETS / "olist_customers_dataset.csv")
orders = pd.read_csv(DATASETS / "olist_orders_dataset.csv")
items = pd.read_csv(DATASETS / "olist_order_items_dataset.csv")

key_checks = pd.DataFrame([
    {"table": "customers", "candidate_key": "customer_id", "rows": len(customers), "unique": customers["customer_id"].nunique()},
    {"table": "customers", "candidate_key": "customer_unique_id", "rows": len(customers), "unique": customers["customer_unique_id"].nunique()},
    {"table": "orders", "candidate_key": "order_id", "rows": len(orders), "unique": orders["order_id"].nunique()},
    {"table": "order_items", "candidate_key": "order_id + order_item_id", "rows": len(items), "unique": items[["order_id", "order_item_id"]].drop_duplicates().shape[0]},
])
display(key_checks)

repeat_id_count = customers.groupby("customer_unique_id")["customer_id"].nunique()
print(f"Canonical customers with more than one Olist customer_id: {(repeat_id_count > 1).sum():,}")

,table,candidate_key,rows,unique
0,customers,customer_id,99441,99441
1,customers,customer_unique_id,99441,96096
2,orders,order_id,99441,99441
3,order_items,order_id + order_item_id,112650,112650


Canonical customers with more than one Olist customer_id: 2,997


## Audit conclusions

- Olist is the only customer-level source of truth.
- `customer_unique_id` becomes `customer_id` in the warehouse.
- Clickstream links are simulated and used only for behavioral demonstration.
- Amazon reviews remain an external product-intelligence module.
- No age, discount, or support-ticket source exists, so the project must not claim age analysis, discount analysis, or real complaint-ticket analytics.